[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/02_Introduction_to_ONNX/04_Installation_and_Setup/Installation_and_Setup_Deep_Dive.ipynb)

# 2.4 Installation and Setup — Deep Dive

Set up a complete ONNX development environment with proper package management, verification, and troubleshooting.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Package Landscape](#section-1) | Core and optional ONNX packages |
| 2 | [Installation Methods](#section-2) | pip, conda, and source builds |
| 3 | [GPU Setup](#section-3) | CUDA, cuDNN, and onnxruntime-gpu |
| 4 | [Verification Suite](#section-4) | Complete installation verification |
| 5 | [End-to-End Smoke Test](#section-5) | Build, validate, and run a model |
| 6 | [Version Compatibility Matrix](#section-6) | Matching package versions |
| 7 | [Troubleshooting Guide](#section-7) | Common errors and solutions |
| 8 | [Development Environment Setup](#section-8) | IDE, linting, and tooling |
| 9 | [Key Takeaways](#section-9) | Summary |

### Prerequisites

- Python 3.8+ installed
- pip or conda package manager
- (Optional) NVIDIA GPU with CUDA for GPU acceleration

<a id='section-1'></a>
## Section 1: Package Landscape

### Core Packages

| Package | Purpose | Required? |
|---------|---------|:---------:|
| `onnx` | Create, load, validate, inspect ONNX models | Yes |
| `onnxruntime` | CPU inference engine | Yes |
| `numpy` | Array operations, test data generation | Yes |
| `protobuf` | Serialization backend (installed with onnx) | Auto |

### Optional Packages

| Package | Purpose | When Needed |
|---------|---------|------------|
| `onnxruntime-gpu` | GPU inference (replaces CPU version) | NVIDIA GPU deployment |
| `onnxoptimizer` | Standalone graph optimization | Advanced optimization |
| `onnx-simplifier` | Simplify/clean exported models | Post-export cleanup |
| `torch` | PyTorch model export | `torch.onnx.export()` |
| `tf2onnx` | TensorFlow model conversion | TF/Keras → ONNX |
| `skl2onnx` | scikit-learn conversion | sklearn → ONNX |
| `netron` | Visual model inspection | Debugging/documentation |
| `matplotlib` | Plotting and visualization | Tutorial exercises |

### Package Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                     Your Python Code                        │
├───────────────┬───────────────┬─────────────────────────────┤
│   onnx        │  onnxruntime  │  Framework converters       │
│   (create +   │  (execute)    │  (torch.onnx, tf2onnx,     │
│    validate)  │               │   skl2onnx)                 │
├───────────────┼───────────────┼─────────────────────────────┤
│   protobuf    │  C++ runtime  │  Framework libs             │
│   (serialize) │  + EPs        │  (PyTorch, TF, sklearn)     │
└───────────────┴───────────────┴─────────────────────────────┘
```

<a id='section-2'></a>
## Section 2: Installation Methods

### Method 1: pip (Recommended)

```bash
# CPU-only (most common for tutorials and development)
pip install onnx onnxruntime numpy

# GPU (requires matching CUDA toolkit)
pip install onnx onnxruntime-gpu numpy

# Full development environment
pip install onnx onnxruntime numpy matplotlib
```

### Method 2: conda

```bash
conda install -c conda-forge onnx onnxruntime numpy
```

### Method 3: Virtual Environment (Best Practice)

```bash
python -m venv onnx_env
source onnx_env/bin/activate      # Linux/macOS
onnx_env\Scripts\activate          # Windows
pip install onnx onnxruntime numpy matplotlib
```

### Important: onnxruntime vs onnxruntime-gpu

These packages are **mutually exclusive** — installing one uninstalls the other:

$$\text{onnxruntime} \oplus \text{onnxruntime-gpu} \quad \text{(XOR — install only one)}$$

<a id='section-3'></a>
## Section 3: GPU Setup

### CUDA Compatibility

For GPU inference, you need matching versions of:

```
Driver ≥ CUDA Toolkit ≥ onnxruntime-gpu wheel

  NVIDIA Driver (e.g., 535.x)
       │
       ▼
  CUDA Toolkit (e.g., 12.x)
       │
       ▼
  cuDNN (e.g., 8.9)
       │
       ▼
  onnxruntime-gpu (matches CUDA version)
```

### Version Matrix

| onnxruntime-gpu | CUDA | cuDNN | Python |
|:-:|:-:|:-:|:-:|
| 1.16+ | 11.8 or 12.x | 8.x | 3.8–3.12 |
| 1.14–1.15 | 11.8 | 8.x | 3.8–3.11 |
| 1.12–1.13 | 11.6 | 8.x | 3.7–3.10 |

<a id='section-4'></a>
## Section 4: Verification Suite

Run these checks after installation to verify everything works.

In [ ]:
import sys

print('Installation Verification Report')
print('=' * 60)
print(f'Python: {sys.version}')
print(f'Platform: {sys.platform}')

# Check core packages
packages = {
    'onnx': 'Model creation and validation',
    'onnxruntime': 'Inference engine',
    'numpy': 'Array operations',
    'google.protobuf': 'Serialization backend',
}

print(f'\nCore Packages:')
all_ok = True
for pkg, desc in packages.items():
    try:
        mod = __import__(pkg)
        version = getattr(mod, '__version__', 'unknown')
        print(f'  [OK] {pkg:20s} v{version:15s} ({desc})')
    except ImportError:
        print(f'  [!!] {pkg:20s} NOT INSTALLED     ({desc})')
        all_ok = False

# Check optional packages
optional = ['matplotlib', 'torch', 'tensorflow', 'sklearn']
print(f'\nOptional Packages:')
for pkg in optional:
    try:
        mod = __import__(pkg)
        version = getattr(mod, '__version__', 'unknown')
        print(f'  [OK] {pkg:20s} v{version}')
    except ImportError:
        print(f'  [--] {pkg:20s} not installed (optional)')

In [ ]:
import onnx
import onnxruntime as ort
import numpy as np

# Check ONNX capabilities
print('ONNX Capabilities:')
print(f'  IR version support: up to {onnx.IR_VERSION}')
print(f'  Has parser:         {hasattr(onnx, "parser")}')
print(f'  Has shape_inference: {hasattr(onnx, "shape_inference")}')
print(f'  Has checker:        {hasattr(onnx, "checker")}')

# Check ORT capabilities
print(f'\nONNX Runtime Capabilities:')
providers = ort.get_available_providers()
print(f'  Available EPs: {providers}')
print(f'  GPU available: {"CUDAExecutionProvider" in providers}')
print(f'  TensorRT:      {"TensorRTExecutionProvider" in providers}')
print(f'  OpenVINO:      {"OpenVINOExecutionProvider" in providers}')

<a id='section-5'></a>
## Section 5: End-to-End Smoke Test

The definitive test: build a model from scratch, validate it, run inference, and verify results.

In [ ]:
from onnx import TensorProto, numpy_helper
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid)
from onnx.checker import check_model
from onnx import shape_inference

print('End-to-End Smoke Test')
print('=' * 60)

# Step 1: Build model Y = ReLU(X @ W + b)
W_data = np.array([[0.5, -0.3, 0.1],
                   [0.2,  0.8, -0.4]], dtype=np.float32)
b_data = np.array([0.1, -0.1, 0.0], dtype=np.float32)

W_init = numpy_helper.from_array(W_data, name='W')
b_init = numpy_helper.from_array(b_data, name='b')

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 2])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 3])

nodes = [
    make_node('MatMul', ['X', 'W'], ['XW']),
    make_node('Add', ['XW', 'b'], ['pre_act']),
    make_node('Relu', ['pre_act'], ['Y']),
]

graph = make_graph(nodes, 'smoke_test', [X], [Y], [W_init, b_init])
model = make_model(graph, opset_imports=[make_opsetid('', 18)])
print('[1] Build:    OK (3 nodes, 2 initializers)')

# Step 2: Validate
check_model(model)
print('[2] Validate: OK (check_model passed)')

# Step 3: Shape inference
inferred = shape_inference.infer_shapes(model)
n_inferred = len(inferred.graph.value_info)
print(f'[3] Shapes:   OK ({n_inferred} intermediate shapes inferred)')

# Step 4: Serialize and deserialize
model_bytes = model.SerializeToString()
model_loaded = onnx.load_model_from_string(model_bytes)
roundtrip_ok = model_bytes == model_loaded.SerializeToString()
print(f'[4] Serde:    OK (round-trip: {roundtrip_ok}, size: {len(model_bytes)} bytes)')

# Step 5: Run inference
sess = ort.InferenceSession(
    model_bytes, providers=['CPUExecutionProvider'])

x = np.array([[1.0, 2.0], [3.0, 4.0]], dtype=np.float32)
result = sess.run(None, {'X': x})[0]
expected = np.maximum(x @ W_data + b_data, 0)
match = np.allclose(result, expected)
print(f'[5] Inference: OK (output matches NumPy: {match})')

print(f'\n{"="*60}')
print(f'ALL CHECKS PASSED — Installation is working correctly!')
print(f'{"="*60}')

<a id='section-6'></a>
## Section 6: Version Compatibility Matrix

### Matching onnx and onnxruntime

The `onnx` package and `onnxruntime` have **independent version numbers** but must be compatible. Generally, a newer onnxruntime supports all older onnx versions.

### Compatibility Rule

$$\text{onnxruntime\_version} \geq \text{min\_ort\_for}(\text{onnx\_version})$$

### Practical Guidance

- Always install the **latest compatible versions** of both
- If you pin `onnx`, check the onnxruntime release notes for minimum `onnx` requirements
- Use `pip install --upgrade onnx onnxruntime` to stay current

<a id='section-7'></a>
## Section 7: Troubleshooting Guide

### Common Issues and Solutions

| Issue | Symptom | Solution |
|-------|---------|--------|
| **protobuf conflict** | `ImportError: libprotobuf.so` | Use clean venv; pin protobuf version |
| **GPU not detected** | `CUDAExecutionProvider` missing | Check CUDA toolkit version matches wheel |
| **ORT version mismatch** | `check_model` passes but ORT fails | Update onnxruntime to match onnx version |
| **Shape errors** | Runtime shape mismatch | Run `shape_inference.infer_shapes()` first |
| **opset too new** | `No opset registered for domain` | Lower opset version or update ORT |
| **Colab issues** | Package conflicts | Restart runtime after installation |

### Diagnostic Commands

```bash
# Check installed versions
pip show onnx onnxruntime protobuf

# Check CUDA (for GPU)
nvidia-smi
nvcc --version

# Test import
python -c "import onnx; import onnxruntime; print('OK')"
```

<a id='section-8'></a>
## Section 8: Development Environment Setup

### Recommended VS Code Extensions

| Extension | Purpose |
|-----------|--------|
| Python | Language support, debugging |
| Jupyter | Notebook support |
| Pylance | Type checking, autocomplete |
| Netron Viewer | Visual ONNX model inspection |

### Project Structure

```
my_onnx_project/
├── models/              ← .onnx files
├── data/                ← test data, calibration data
├── scripts/
│   ├── export.py        ← framework → ONNX
│   ├── optimize.py      ← graph optimization
│   ├── validate.py      ← check + shape inference
│   └── benchmark.py     ← latency/throughput testing
├── tests/               ← numerical correctness tests
├── requirements.txt     ← pinned dependencies
└── README.md
```

<a id='section-9'></a>
## Section 9: Key Takeaways

### Installation Cheat Sheet

```bash
# Minimum (CPU)
pip install onnx onnxruntime numpy

# Full tutorial environment
pip install onnx onnxruntime numpy matplotlib

# GPU
pip install onnx onnxruntime-gpu numpy
```

### Verification Checklist

1. `import onnx` — can create models
2. `import onnxruntime` — can run inference
3. `check_model()` — validation works
4. `InferenceSession.run()` — end-to-end execution works
5. Results match NumPy — numerical correctness verified

### Critical Rules

1. **Use virtual environments** to avoid package conflicts
2. **Never install both** `onnxruntime` and `onnxruntime-gpu`
3. **Match CUDA versions** carefully for GPU setup
4. **Run the smoke test** after every installation

---

**This completes Module 2: Introduction to ONNX.**

**Next:** [Module 3: ONNX Architecture and Internals](../../03_ONNX_Architecture_and_Internals/) — DAG theory, nodes, edges, IR specification, and type system.